In [1]:
import os
os.listdir()

['.config', 'bios.csv', 'sample_data']

In [2]:
import sqlite3
conn = sqlite3.connect(":memory:")

In [3]:
import pandas as pd

bios_raw = pd.read_csv("bios.csv")

bios_raw.to_sql("bios", conn, index= False, if_exists="replace")


70664

In [4]:
pd.read_sql(
    "select name from sqlite_master where type= 'table';",
    conn
)

,name
0,bios


In [5]:
pd.read_sql(
    "select * from bios limit 5;",
    conn
)

,Roles,Sex,Full name,Used name,Born,Died,NOC,athlete_id,Measurements,Affiliations,Nick/petnames,Title(s),Other names,Nationality,Original name,Name order
0,Competed in Olympic Games,Male,"François Joseph Marie Antoine ""Jean-François""•...",Jean-François•Blanchy,"12 December 1886 in Bordeaux, Gironde (FRA)","2 October 1960 in Saint-Jean-de-Luz, Pyrénées-...",France,1.0,None,None,None,None,None,None,None,None
1,Competed in Olympic Games,Male,Arnaud Benjamin•Boetsch,Arnaud•Boetsch,"1 April 1969 in Meulan, Yvelines (FRA)",None,France,2.0,183 cm / 76 kg,"Racing Club de France, Paris (FRA)",None,None,None,None,None,None
2,Competed in Olympic Games • Administrator,Male,Jean Laurent Robert•Borotra,Jean•Borotra,"13 August 1898 in Biarritz, Pyrénées-Atlantiqu...","17 July 1994 in Arbonne, Pyrénées-Atlantiques ...",France,3.0,183 cm / 76 kg,"TCP, Paris (FRA)",Le Basque Bondissant (The Bounding Basque),None,None,None,None,None
3,Competed in Olympic Games,Male,Jacques Marie Stanislas Jean•Brugnon,Jacques•Brugnon,"11 May 1895 in Paris VIIIe, Paris (FRA)","20 March 1978 in Monaco, Monaco (MON)",France,4.0,168 cm / 64 kg,"Sporting club de Paris, Paris (FRA)",Toto,None,None,None,None,None
4,Competed in Olympic Games,Male,Henry Albert•Canet,Albert•Canet,"17 April 1878 in Wandsworth, England (GBR)","25 July 1930 in Paris VIIe, Paris (FRA)",France,5.0,None,"TCP, Paris (FRA)",None,None,None,None,None,None


In [6]:
pd.read_sql(
    "PRAGMA table_info(bios);",
    conn
)

,cid,name,type,notnull,dflt_value,pk
0,0,Roles,TEXT,0,None,0
1,1,Sex,TEXT,0,None,0
2,2,Full name,TEXT,0,None,0
3,3,Used name,TEXT,0,None,0
4,4,Born,TEXT,0,None,0
5,5,Died,TEXT,0,None,0
6,6,NOC,TEXT,0,None,0
7,7,athlete_id,REAL,0,None,0
8,8,Measurements,TEXT,0,None,0
9,9,Affiliations,TEXT,0,None,0


Correcting Columns names by SQL

In [7]:
conn.execute("""
CREATE VIEW bios_cleaner AS
SELECT
    athlete_id                         AS athlete_id,
    Roles                              AS roles,
    Sex                                AS sex,
    "Full name"                        AS full_name,
    "Used name"                        AS used_name,
    Born                               AS born_raw,
    Died                               AS died_raw,
    NOC                                AS noc,
    Measurements                       AS measurements_raw,
    Affiliations                       AS affiliations,
    "Nick/petnames"                    AS nick_petnames,
    "Title(s)"                         AS titles,
    "Other names"                      AS other_names,
    Nationality                        AS nationality,
    "Original name"                    AS original_name,
    "Name order"                       AS name_order
FROM bios;
""")


In [8]:
pd.read_sql(
    """
    SELECT athlete_id, full_name, used_name
    FROM bios_cleaner
    LIMIT 5;
    """,
    conn
)

,athlete_id,full_name,used_name
0,1.0,"François Joseph Marie Antoine ""Jean-François""•...",Jean-François•Blanchy
1,2.0,Arnaud Benjamin•Boetsch,Arnaud•Boetsch
2,3.0,Jean Laurent Robert•Borotra,Jean•Borotra
3,4.0,Jacques Marie Stanislas Jean•Brugnon,Jacques•Brugnon
4,5.0,Henry Albert•Canet,Albert•Canet


In [9]:
pd.read_sql(
    """
    SELECT measurements_raw
    FROM bios_cleaner
    WHERE measurements_raw IS NOT NULL
    LIMIT 5;
    """,
    conn
)

,measurements_raw
0,183 cm / 76 kg
1,183 cm / 76 kg
2,168 cm / 64 kg
3,181 cm / 70 kg
4,180 cm / 73 kg


Understanding the Position. of the string, to fetch it.

In [10]:
pd.read_sql(
    """
SELECT
  measurements_raw,
  instr(measurements_raw, ' cm') AS cm_position
FROM bios_cleaner
LIMIT 5;


    """, conn
)

,measurements_raw,cm_position
0,None,NaN
1,183 cm / 76 kg,4.0
2,183 cm / 76 kg,4.0
3,168 cm / 64 kg,4.0
4,None,NaN


Fetching, converting Measurement from string to int

In [11]:
pd.read_sql(
    """
SELECT
  athlete_id, measurements_raw,

  CASE
    WHEN measurements_raw LIKE "%cm%"
    THEN CAST(
      substr(measurements_raw, 1, instr(measurements_raw, " cm") -1)
      AS INTEGER
    )
    ELSE NULL
  END AS height_cm


FROM bios_cleaner
LIMIT 5;


    """, conn
)

,athlete_id,measurements_raw,height_cm
0,1.0,None,NaN
1,2.0,183 cm / 76 kg,183.0
2,3.0,183 cm / 76 kg,183.0
3,4.0,168 cm / 64 kg,168.0
4,5.0,None,NaN


Understanding the Position. of the string, to fetch it.

In [12]:
pd.read_sql(
    """
SELECT
  athlete_id, measurements_raw,


  instr(measurements_raw, '/') AS slash_position,
  instr(measurements_raw, ' kg') AS kg_position
FROM bios_cleaner
WHERE measurements_raw IS NOT NULL
LIMIT 5;


    """, conn
)

,athlete_id,measurements_raw,slash_position,kg_position
0,2.0,183 cm / 76 kg,8,12
1,3.0,183 cm / 76 kg,8,12
2,4.0,168 cm / 64 kg,8,12
3,6.0,181 cm / 70 kg,8,12
4,7.0,180 cm / 73 kg,8,12


Fetching, converting Measurement from string to int

In [13]:
pd.read_sql(
    """
    SELECT athlete_id, measurements_raw,

    CASE
      WHEN measurements_raw LIKE "%kg%"

      THEN CAST(
        substr(measurements_raw,
        instr(measurements_raw, "/") + 2,
        instr(measurements_raw, " kg") - instr(measurements_raw, "/") -2
        )
        AS INTEGER
      )
      ELSE NULL
    END AS weight_kg
  FROM bios_cleaner
  LIMIT 10;
    """, conn
)

,athlete_id,measurements_raw,weight_kg
0,1.0,None,NaN
1,2.0,183 cm / 76 kg,76.0
2,3.0,183 cm / 76 kg,76.0
3,4.0,168 cm / 64 kg,64.0
4,5.0,None,NaN
5,6.0,181 cm / 70 kg,70.0
6,7.0,180 cm / 73 kg,73.0
7,8.0,None,NaN
8,9.0,None,NaN
9,10.0,None,NaN


In [14]:
pd.read_sql(
    """
  SELECT born_raw
  FROM bios_cleaner
  WHERE born_raw IS NOT NULL

  LIMIT 10;
    """, conn
)

,born_raw
0,"12 December 1886 in Bordeaux, Gironde (FRA)"
1,"1 April 1969 in Meulan, Yvelines (FRA)"
2,"13 August 1898 in Biarritz, Pyrénées-Atlantiqu..."
3,"11 May 1895 in Paris VIIIe, Paris (FRA)"
4,"17 April 1878 in Wandsworth, England (GBR)"
5,"13 January 1970 in Amiens, Somme (FRA)"
6,"27 November 1969 in Ris-Orangis, Essonne (FRA)"
7,"14 December 1901 in Villeurbanne, Rhône (FRA)"
8,"4 August 1896 in Nîmes, Gard (FRA)"
9,"16 July 1868 in Farges-Allichamps, Cher (FRA)"


Fetching Birth Date

In [15]:
pd.read_sql(
    """
  SELECT born_raw,

  CASE
    WHEN
      born_raw LIKE "% in %"
    THEN
      substr(
        born_raw,
        1,
        instr(born_raw, " in ") -1
      )
      ELSE NULL
    END AS birth_date_raw
  FROM bios_cleaner
  WHERE born_raw IS NOT NULL

  LIMIT 10;
    """, conn
)

,born_raw,birth_date_raw
0,"12 December 1886 in Bordeaux, Gironde (FRA)",12 December 1886
1,"1 April 1969 in Meulan, Yvelines (FRA)",1 April 1969
2,"13 August 1898 in Biarritz, Pyrénées-Atlantiqu...",13 August 1898
3,"11 May 1895 in Paris VIIIe, Paris (FRA)",11 May 1895
4,"17 April 1878 in Wandsworth, England (GBR)",17 April 1878
5,"13 January 1970 in Amiens, Somme (FRA)",13 January 1970
6,"27 November 1969 in Ris-Orangis, Essonne (FRA)",27 November 1969
7,"14 December 1901 in Villeurbanne, Rhône (FRA)",14 December 1901
8,"4 August 1896 in Nîmes, Gard (FRA)",4 August 1896
9,"16 July 1868 in Farges-Allichamps, Cher (FRA)",16 July 1868


Converting String Year into Date format

In [16]:
pd.read_sql(
    """
   SELECT
    birth_date_raw,

    DATE(
        printf(
            '%04d-%02d-%02d',
            CAST(substr(birth_date_raw, -4) AS INTEGER),
            CASE
                WHEN birth_date_raw LIKE '%January%' THEN 1
                WHEN birth_date_raw LIKE '%February%' THEN 2
                WHEN birth_date_raw LIKE '%March%' THEN 3
                WHEN birth_date_raw LIKE '%April%' THEN 4
                WHEN birth_date_raw LIKE '%May%' THEN 5
                WHEN birth_date_raw LIKE '%June%' THEN 6
                WHEN birth_date_raw LIKE '%July%' THEN 7
                WHEN birth_date_raw LIKE '%August%' THEN 8
                WHEN birth_date_raw LIKE '%September%' THEN 9
                WHEN birth_date_raw LIKE '%October%' THEN 10
                WHEN birth_date_raw LIKE '%November%' THEN 11
                WHEN birth_date_raw LIKE '%December%' THEN 12
            END,
            CAST(substr(birth_date_raw, 1, instr(birth_date_raw, ' ') - 1) AS INTEGER)
        )
    ) AS birth_date

FROM (
    SELECT
        substr(born_raw, 1, instr(born_raw, ' in ') - 1) AS birth_date_raw
    FROM bios_cleaner
    WHERE born_raw LIKE '% in %'
)
LIMIT 5;


    """, conn
)

,birth_date_raw,birth_date
0,12 December 1886,1886-12-12
1,1 April 1969,1969-04-01
2,13 August 1898,1898-08-13
3,11 May 1895,1895-05-11
4,17 April 1878,1878-04-17


Extract birth_country_code
Observation

Country code is always inside parentheses at the end:

(FRA)
(GBR)

Extracting and Saperating born_raw into different columns

In [17]:
pd.read_sql(
"""
SELECT
    born_raw,

    --City
    substr(
        born_raw,
        instr(born_raw, ' in ')+4,
        instr(born_raw, ',') - instr(born_raw, ' in ') -4
    ) AS birth_city,

    --Region
    substr(
            born_raw,
            instr(born_raw, ', ') + 2,
            instr(born_raw, ' (') - instr(born_raw, ', ') - 1
        )AS birth_region,

    --Country Code
    substr(
            born_raw,
            instr(born_raw, '(') + 1,
            instr(born_raw, ')') - instr(born_raw, '(') -1
          ) AS birth_country_code

FROM bios_cleaner
WHERE born_raw LIKE '% in %'
LIMIT 5;
""",conn
)


,born_raw,birth_city,birth_region,birth_country_code
0,"12 December 1886 in Bordeaux, Gironde (FRA)",Bordeaux,Gironde,FRA
1,"1 April 1969 in Meulan, Yvelines (FRA)",Meulan,Yvelines,FRA
2,"13 August 1898 in Biarritz, Pyrénées-Atlantiqu...",Biarritz,Pyrénées-Atlantiques,FRA
3,"11 May 1895 in Paris VIIIe, Paris (FRA)",Paris VIIIe,Paris,FRA
4,"17 April 1878 in Wandsworth, England (GBR)",Wandsworth,England,GBR


In [18]:
conn.executescript("""
DROP VIEW IF EXISTS bios_birth_clean_v4;

CREATE VIEW bios_birth_clean_v4 AS
SELECT
    athlete_id,
    born_raw,

    /* -------------------------
       Date-only part
    --------------------------*/
    substr(born_raw, 1, instr(born_raw, ' in ') - 1) AS birth_date_raw,

    /* -------------------------
       Proper DATE (FIXED YEAR)
    --------------------------*/
    DATE(
        printf(
            '%04d-%02d-%02d',

            -- YEAR (from date-only string)
            CAST(
                substr(
                    substr(born_raw, 1, instr(born_raw, ' in ') - 1),
                    -4
                ) AS INTEGER
            ),

            -- MONTH
            CASE
                WHEN born_raw LIKE '%January%' THEN 1
                WHEN born_raw LIKE '%February%' THEN 2
                WHEN born_raw LIKE '%March%' THEN 3
                WHEN born_raw LIKE '%April%' THEN 4
                WHEN born_raw LIKE '%May%' THEN 5
                WHEN born_raw LIKE '%June%' THEN 6
                WHEN born_raw LIKE '%July%' THEN 7
                WHEN born_raw LIKE '%August%' THEN 8
                WHEN born_raw LIKE '%September%' THEN 9
                WHEN born_raw LIKE '%October%' THEN 10
                WHEN born_raw LIKE '%November%' THEN 11
                WHEN born_raw LIKE '%December%' THEN 12
            END,

            -- DAY
            CAST(
                substr(
                    substr(born_raw, 1, instr(born_raw, ' in ') - 1),
                    1,
                    instr(substr(born_raw, 1, instr(born_raw, ' in ') - 1), ' ') - 1
                ) AS INTEGER
            )
        )
    ) AS birth_date,

    /* -------------------------
       City
    --------------------------*/
    substr(
        born_raw,
        instr(born_raw, ' in ') + 4,
        instr(born_raw, ',') - instr(born_raw, ' in ') - 4
    ) AS birth_city,

    /* -------------------------
       Region
    --------------------------*/
    substr(
        born_raw,
        instr(born_raw, ', ') + 2,
        instr(born_raw, ' (') - instr(born_raw, ', ') - 2
    ) AS birth_region,

    /* -------------------------
       Country code
    --------------------------*/
    substr(
        born_raw,
        instr(born_raw, '(') + 1,
        instr(born_raw, ')') - instr(born_raw, '(') - 1
    ) AS birth_country_code

FROM bios_cleaner
WHERE born_raw LIKE '% in %';
""")


In [19]:
pd.read_sql(
    """
    SELECT *
    FROM bios_birth_clean_v4
    LIMIT 10
    """,
    conn
)

,athlete_id,born_raw,birth_date_raw,birth_date,birth_city,birth_region,birth_country_code
0,1.0,"12 December 1886 in Bordeaux, Gironde (FRA)",12 December 1886,1886-12-12,Bordeaux,Gironde,FRA
1,2.0,"1 April 1969 in Meulan, Yvelines (FRA)",1 April 1969,1969-04-01,Meulan,Yvelines,FRA
2,3.0,"13 August 1898 in Biarritz, Pyrénées-Atlantiqu...",13 August 1898,1898-08-13,Biarritz,Pyrénées-Atlantiques,FRA
3,4.0,"11 May 1895 in Paris VIIIe, Paris (FRA)",11 May 1895,1895-05-11,Paris VIIIe,Paris,FRA
4,5.0,"17 April 1878 in Wandsworth, England (GBR)",17 April 1878,1878-04-17,Wandsworth,England,GBR
5,6.0,"13 January 1970 in Amiens, Somme (FRA)",13 January 1970,1970-01-13,Amiens,Somme,FRA
6,7.0,"27 November 1969 in Ris-Orangis, Essonne (FRA)",27 November 1969,1969-11-27,Ris-Orangis,Essonne,FRA
7,8.0,"14 December 1901 in Villeurbanne, Rhône (FRA)",14 December 1901,1901-12-14,Villeurbanne,Rhône,FRA
8,9.0,"4 August 1896 in Nîmes, Gard (FRA)",4 August 1896,1896-08-04,Nîmes,Gard,FRA
9,10.0,"16 July 1868 in Farges-Allichamps, Cher (FRA)",16 July 1868,1868-07-16,Farges-Allichamps,Cher,FRA


Creating Table from View for Birth Data

In [20]:
conn.executescript("""
DROP TABLE IF EXISTS bios_birth_final;
CREATE TABLE bios_birth_final AS
SELECT *
FROM bios_birth_clean_v4;
""")

In [21]:
pd.read_sql("""
SELECT *
FROM bios_birth_final
LIMIT 5;

""",conn)

,athlete_id,born_raw,birth_date_raw,birth_date,birth_city,birth_region,birth_country_code
0,1.0,"12 December 1886 in Bordeaux, Gironde (FRA)",12 December 1886,1886-12-12,Bordeaux,Gironde,FRA
1,2.0,"1 April 1969 in Meulan, Yvelines (FRA)",1 April 1969,1969-04-01,Meulan,Yvelines,FRA
2,3.0,"13 August 1898 in Biarritz, Pyrénées-Atlantiqu...",13 August 1898,1898-08-13,Biarritz,Pyrénées-Atlantiques,FRA
3,4.0,"11 May 1895 in Paris VIIIe, Paris (FRA)",11 May 1895,1895-05-11,Paris VIIIe,Paris,FRA
4,5.0,"17 April 1878 in Wandsworth, England (GBR)",17 April 1878,1878-04-17,Wandsworth,England,GBR


Creating the View in the Table

In [22]:
conn.execute("""
CREATE VIEW bios_died_clean AS
SELECT
    athlete_id,
    died_raw,

    -- death_city
    substr(
        died_raw,
        instr(died_raw, ' in ') + 4,
        instr(died_raw, ',') - instr(died_raw, ' in ') - 4
    ) AS death_city,

    -- death_region
    CASE
        WHEN died_raw LIKE '%, % (%)'
        THEN substr(
            died_raw,
            instr(died_raw, ', ') + 2,
            instr(died_raw, ' (') - instr(died_raw, ', ') - 1
        )
        WHEN died_raw LIKE '%, %'
        THEN substr(
            died_raw,
            instr(died_raw, ', ') + 2
        )
        ELSE NULL
    END AS death_region,

    -- death_country_code (optional)
    CASE
        WHEN died_raw LIKE '%(%)'
        THEN substr(
            died_raw,
            instr(died_raw, '(') + 1,
            instr(died_raw, ')') - instr(died_raw, '(') - 1
        )
        ELSE NULL
    END AS death_country_code

FROM bios_cleaner
WHERE died_raw LIKE '% in %';
""")


In [23]:
pd.read_sql(
    """
    SELECT *
    FROM bios_died_clean
    LIMIT 10
    """,
    conn
)

,athlete_id,died_raw,death_city,death_region,death_country_code
0,1.0,"2 October 1960 in Saint-Jean-de-Luz, Pyrénées-...",Saint-Jean-de-Luz,Pyrénées-Atlantiques,FRA
1,3.0,"17 July 1994 in Arbonne, Pyrénées-Atlantiques ...",Arbonne,Pyrénées-Atlantiques,FRA
2,4.0,"20 March 1978 in Monaco, Monaco (MON)",Monaco,Monaco,MON
3,5.0,"25 July 1930 in Paris VIIe, Paris (FRA)",Paris VIIe,Paris,FRA
4,8.0,"2 April 1987 in Saint-Germain-en-Laye, Yveline...",Saint-Germain-en-Laye,Yvelines,FRA
5,9.0,"1 August 1986 in Nîmes, Gard (FRA)",Nîmes,Gard,FRA
6,11.0,"18 November 1932 in Castres, Tarn (FRA)",Castres,Tarn,FRA
7,12.0,"6 September 1978 in Biot, Alpes-Maritimes (FRA)",Biot,Alpes-Maritimes,FRA
8,16.0,"6 August 1965 in Cannes, Alpes-Maritimes (FRA)",Cannes,Alpes-Maritimes,FRA
9,19.0,"6 August 1958 in Vichy, Allier (FRA)",Vichy,Allier,FRA


Creating Table for the Birth_date_cleaned and Death_date-cleaned from the Views that has been created above

Creating Table for Died

In [24]:
conn.execute("""
CREATE TABLE bios_died_final AS
SELECT *
FROM bios_died_clean;
""")


In [25]:
pd.read_sql(
    "SELECT * FROM bios_birth_final LIMIT 5;",
    conn
)

,athlete_id,born_raw,birth_date_raw,birth_date,birth_city,birth_region,birth_country_code
0,1.0,"12 December 1886 in Bordeaux, Gironde (FRA)",12 December 1886,1886-12-12,Bordeaux,Gironde,FRA
1,2.0,"1 April 1969 in Meulan, Yvelines (FRA)",1 April 1969,1969-04-01,Meulan,Yvelines,FRA
2,3.0,"13 August 1898 in Biarritz, Pyrénées-Atlantiqu...",13 August 1898,1898-08-13,Biarritz,Pyrénées-Atlantiques,FRA
3,4.0,"11 May 1895 in Paris VIIIe, Paris (FRA)",11 May 1895,1895-05-11,Paris VIIIe,Paris,FRA
4,5.0,"17 April 1878 in Wandsworth, England (GBR)",17 April 1878,1878-04-17,Wandsworth,England,GBR


In [26]:
pd.read_sql(
    "SELECT * FROM bios_died_final LIMIT 5;",
    conn
)

,athlete_id,died_raw,death_city,death_region,death_country_code
0,1.0,"2 October 1960 in Saint-Jean-de-Luz, Pyrénées-...",Saint-Jean-de-Luz,Pyrénées-Atlantiques,FRA
1,3.0,"17 July 1994 in Arbonne, Pyrénées-Atlantiques ...",Arbonne,Pyrénées-Atlantiques,FRA
2,4.0,"20 March 1978 in Monaco, Monaco (MON)",Monaco,Monaco,MON
3,5.0,"25 July 1930 in Paris VIIe, Paris (FRA)",Paris VIIe,Paris,FRA
4,8.0,"2 April 1987 in Saint-Germain-en-Laye, Yveline...",Saint-Germain-en-Laye,Yvelines,FRA


In [27]:
conn.execute("""DROP VIEW IF EXISTS bios_measurements_clean;""")


Created View for Measurements

In [28]:
conn.execute(
    """CREATE VIEW bios_measurements_clean AS
SELECT
    athlete_id,

    -- extract height
    CASE
        WHEN measurements_raw LIKE '%/%'
        THEN CAST(
            substr(measurements_raw, 1, instr(measurements_raw, '/') - 1)
            AS INTEGER
        )
        ELSE NULL
    END AS height_cm,

    -- extract weight
    CASE
        WHEN measurements_raw LIKE '%/%'
        THEN CAST(
            substr(
                measurements_raw,
                instr(measurements_raw, '/') + 1
            )
            AS INTEGER
        )
        ELSE NULL
    END AS weight_kg

FROM bios_cleaner;

"""
)

In [29]:
conn.execute("""DROP TABLE IF EXISTS bios_measurements_final;""")


In [30]:
conn.execute("""
CREATE TABLE bios_measurements_final AS
SELECT *
FROM bios_measurements_clean;
""")


In [31]:
pd.read_sql(
    "SELECT * FROM bios_measurements_final LIMIT 5;",
    conn
)

,athlete_id,height_cm,weight_kg
0,1.0,NaN,NaN
1,2.0,183.0,76.0
2,3.0,183.0,76.0
3,4.0,168.0,64.0
4,5.0,NaN,NaN


In [32]:
df = pd.read_csv("/content/bios.csv")

In [33]:
df["user_name_clean"] = (
    df["Used name"]
    .astype(str)
    .str.replace(r"[•·‧]", "", regex=True)
    .str.replace(r"([A-Z])([a-z])",r"\1 \2", regex=True)
    .str.strip()
)

In [34]:
df = pd.read_sql(
    "SELECT * FROM bios_cleaner",
    conn
)

In [35]:
pd.read_sql(
    """SELECT name, type
FROM sqlite_master
WHERE name = 'bios_cleaner';
""",
    conn
)

,name,type
0,bios_cleaner,view


In [36]:
pd.read_sql(
    """PRAGMA table_info(bios_cleaner);
""",conn
)

,cid,name,type,notnull,dflt_value,pk
0,0,athlete_id,REAL,0,None,0
1,1,roles,TEXT,0,None,0
2,2,sex,TEXT,0,None,0
3,3,full_name,TEXT,0,None,0
4,4,used_name,TEXT,0,None,0
5,5,born_raw,TEXT,0,None,0
6,6,died_raw,TEXT,0,None,0
7,7,noc,TEXT,0,None,0
8,8,measurements_raw,TEXT,0,None,0
9,9,affiliations,TEXT,0,None,0


In [37]:
import pandas as pd

df = pd.read_sql("SELECT * FROM bios_cleaner;", conn)


In [38]:
df["used_name_clean"] = (
    df["used_name"]
      .astype(str)
      .str.replace(".", " ", regex=False)
      .str.replace("·", " ", regex=False)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)


In [39]:
pd.read_sql(
    """PRAGMA table_info(bios_cleaner);
""",conn
)

,cid,name,type,notnull,dflt_value,pk
0,0,athlete_id,REAL,0,None,0
1,1,roles,TEXT,0,None,0
2,2,sex,TEXT,0,None,0
3,3,full_name,TEXT,0,None,0
4,4,used_name,TEXT,0,None,0
5,5,born_raw,TEXT,0,None,0
6,6,died_raw,TEXT,0,None,0
7,7,noc,TEXT,0,None,0
8,8,measurements_raw,TEXT,0,None,0
9,9,affiliations,TEXT,0,None,0


Creating, new table dim_athlete

In [40]:
conn.execute(
    """DROP VIEW IF EXISTS dim_athlete_clean;""")



Creating View

In [41]:
conn.execute(
    """
    CREATE VIEW dim_athlete_clean AS
    SELECT
    athlete_id,
    full_name,
    used_name,
    REPLACE(used_name, '.', '') AS used_name_clean,
    sex,
    noc
FROM bios_cleaner;
    """
)

Checking if the View is created or Not

In [42]:
pd.read_sql("""SELECT *
FROM dim_athlete_clean
LIMIT 10;""", conn)


,athlete_id,full_name,used_name,used_name_clean,sex,noc
0,1.0,"François Joseph Marie Antoine ""Jean-François""•...",Jean-François•Blanchy,Jean-François•Blanchy,Male,France
1,2.0,Arnaud Benjamin•Boetsch,Arnaud•Boetsch,Arnaud•Boetsch,Male,France
2,3.0,Jean Laurent Robert•Borotra,Jean•Borotra,Jean•Borotra,Male,France
3,4.0,Jacques Marie Stanislas Jean•Brugnon,Jacques•Brugnon,Jacques•Brugnon,Male,France
4,5.0,Henry Albert•Canet,Albert•Canet,Albert•Canet,Male,France
5,6.0,Nicolas•Chatelain,Nicolas•Chatelain,Nicolas•Chatelain,Male,France
6,7.0,Patrick•Chila,Patrick•Chila,Patrick•Chila,Male,France
7,8.0,Henri Jean•Cochet,Henri•Cochet,Henri•Cochet,Male,France
8,9.0,Marcel•Cousin,Marcel•Cousin,Marcel•Cousin,Male,France
9,10.0,Luc Henri Hervé Guy•Gardye de la Chapelle,Guy•de la Chapelle,Guy•de la Chapelle,Male,France


In [43]:
pd.read_sql(
  """
    SELECT COUNT(*) AS Total_rows,
    COUNT (athlete_id) AS distinct_athletes
    FROM dim_athlete_clean;
  """,conn
)


,Total_rows,distinct_athletes
0,70664,70663


In [44]:
conn.execute("""
CREATE TABLE dim_athlete AS
SELECT *
FROM dim_athlete_clean;
""")


In [45]:
pd.read_sql(
    "SELECT * FROM dim_athlete LIMIT 10;",
    conn
)


,athlete_id,full_name,used_name,used_name_clean,sex,noc
0,1.0,"François Joseph Marie Antoine ""Jean-François""•...",Jean-François•Blanchy,Jean-François•Blanchy,Male,France
1,2.0,Arnaud Benjamin•Boetsch,Arnaud•Boetsch,Arnaud•Boetsch,Male,France
2,3.0,Jean Laurent Robert•Borotra,Jean•Borotra,Jean•Borotra,Male,France
3,4.0,Jacques Marie Stanislas Jean•Brugnon,Jacques•Brugnon,Jacques•Brugnon,Male,France
4,5.0,Henry Albert•Canet,Albert•Canet,Albert•Canet,Male,France
5,6.0,Nicolas•Chatelain,Nicolas•Chatelain,Nicolas•Chatelain,Male,France
6,7.0,Patrick•Chila,Patrick•Chila,Patrick•Chila,Male,France
7,8.0,Henri Jean•Cochet,Henri•Cochet,Henri•Cochet,Male,France
8,9.0,Marcel•Cousin,Marcel•Cousin,Marcel•Cousin,Male,France
9,10.0,Luc Henri Hervé Guy•Gardye de la Chapelle,Guy•de la Chapelle,Guy•de la Chapelle,Male,France


In [46]:
pd.read_sql("""PRAGMA table_info(athlete_birth_final);""",conn)


,cid,name,type,notnull,dflt_value,pk


In [47]:
pd.read_sql("""PRAGMA table_info(bios_cleaner);""",conn)


,cid,name,type,notnull,dflt_value,pk
0,0,athlete_id,REAL,0,None,0
1,1,roles,TEXT,0,None,0
2,2,sex,TEXT,0,None,0
3,3,full_name,TEXT,0,None,0
4,4,used_name,TEXT,0,None,0
5,5,born_raw,TEXT,0,None,0
6,6,died_raw,TEXT,0,None,0
7,7,noc,TEXT,0,None,0
8,8,measurements_raw,TEXT,0,None,0
9,9,affiliations,TEXT,0,None,0


In [48]:
pd.read_sql(
    """
    SELECT used_name_clean
    FROM dim_athlete
    WHERE used_name_clean LIKE '%.%';
    """,
    conn
)


,used_name_clean


In [49]:
pd.read_sql(
    "SELECT full_name, used_name_clean FROM dim_athlete LIMIT 10;",
    conn
)


,full_name,used_name_clean
0,"François Joseph Marie Antoine ""Jean-François""•...",Jean-François•Blanchy
1,Arnaud Benjamin•Boetsch,Arnaud•Boetsch
2,Jean Laurent Robert•Borotra,Jean•Borotra
3,Jacques Marie Stanislas Jean•Brugnon,Jacques•Brugnon
4,Henry Albert•Canet,Albert•Canet
5,Nicolas•Chatelain,Nicolas•Chatelain
6,Patrick•Chila,Patrick•Chila
7,Henri Jean•Cochet,Henri•Cochet
8,Marcel•Cousin,Marcel•Cousin
9,Luc Henri Hervé Guy•Gardye de la Chapelle,Guy•de la Chapelle


Validating full_name Cloumn, Removing • frm the name

In [50]:
conn.executescript("""
DROP VIEW IF EXISTS dim_athlete_clean;

CREATE VIEW dim_athlete_clean AS
SELECT
    athlete_id,
    full_name,

    TRIM(
      REPLACE(
        REPLACE(
          REPLACE(used_name, '·', ' '),
          '•', ' '
        ),
        '‧', ' '
      )
    ) AS used_name_clean,

    sex,
    nationality,
    noc
FROM bios_cleaner;
""")


In [51]:
pd.read_sql(
    """
    SELECT used_name_clean
    FROM dim_athlete_clean
    WHERE used_name_clean LIKE '%·%'
       OR used_name_clean LIKE '%•%'
       OR used_name_clean LIKE '%‧%';
    """,
    conn
)


,used_name_clean


In [52]:
pd.read_sql("""
SELECT name, type
FROM sqlite_master
WHERE type IN ('table','view')
ORDER BY type, name;
""", conn)


,name,type
0,bios,table
1,bios_birth_final,table
2,bios_died_final,table
3,bios_measurements_final,table
4,dim_athlete,table
5,bios_birth_clean_v4,view
6,bios_cleaner,view
7,bios_died_clean,view
8,bios_measurements_clean,view
9,dim_athlete_clean,view


In [53]:
pd.read_sql("""
SELECT *
FROM bios_birth_clean_v4
LIMIT 10;
""", conn)

,athlete_id,born_raw,birth_date_raw,birth_date,birth_city,birth_region,birth_country_code
0,1.0,"12 December 1886 in Bordeaux, Gironde (FRA)",12 December 1886,1886-12-12,Bordeaux,Gironde,FRA
1,2.0,"1 April 1969 in Meulan, Yvelines (FRA)",1 April 1969,1969-04-01,Meulan,Yvelines,FRA
2,3.0,"13 August 1898 in Biarritz, Pyrénées-Atlantiqu...",13 August 1898,1898-08-13,Biarritz,Pyrénées-Atlantiques,FRA
3,4.0,"11 May 1895 in Paris VIIIe, Paris (FRA)",11 May 1895,1895-05-11,Paris VIIIe,Paris,FRA
4,5.0,"17 April 1878 in Wandsworth, England (GBR)",17 April 1878,1878-04-17,Wandsworth,England,GBR
5,6.0,"13 January 1970 in Amiens, Somme (FRA)",13 January 1970,1970-01-13,Amiens,Somme,FRA
6,7.0,"27 November 1969 in Ris-Orangis, Essonne (FRA)",27 November 1969,1969-11-27,Ris-Orangis,Essonne,FRA
7,8.0,"14 December 1901 in Villeurbanne, Rhône (FRA)",14 December 1901,1901-12-14,Villeurbanne,Rhône,FRA
8,9.0,"4 August 1896 in Nîmes, Gard (FRA)",4 August 1896,1896-08-04,Nîmes,Gard,FRA
9,10.0,"16 July 1868 in Farges-Allichamps, Cher (FRA)",16 July 1868,1868-07-16,Farges-Allichamps,Cher,FRA


In [54]:
pd.read_sql("""
SELECT *
FROM bios_measurements_final
LIMIT 10;
""", conn)

,athlete_id,height_cm,weight_kg
0,1.0,NaN,NaN
1,2.0,183.0,76.0
2,3.0,183.0,76.0
3,4.0,168.0,64.0
4,5.0,NaN,NaN
5,6.0,181.0,70.0
6,7.0,180.0,73.0
7,8.0,NaN,NaN
8,9.0,NaN,NaN
9,10.0,NaN,NaN


In [55]:
pd.read_sql("""
SELECT
  athlete_id,
  COUNT(*) AS row_count
FROM dim_athlete
GROUP BY athlete_id
HAVING COUNT(*) > 1;
""",conn)

,athlete_id,row_count


In [56]:
pd.read_sql("""
SELECT
    COUNT(*)                     AS total_rows,
    COUNT(DISTINCT athlete_id)   AS distinct_athletes
FROM dim_athlete_clean;

""",conn)

,total_rows,distinct_athletes
0,70664,70663


In [57]:
pd.read_sql("""PRAGMA table_info(dim_athlete_clean);""",conn)



,cid,name,type,notnull,dflt_value,pk
0,0,athlete_id,REAL,0,None,0
1,1,full_name,TEXT,0,None,0
2,2,used_name_clean,,0,None,0
3,3,sex,TEXT,0,None,0
4,4,nationality,TEXT,0,None,0
5,5,noc,TEXT,0,None,0


How many athletes per country AND per sex?

In [58]:
pd.read_sql("""
SELECT
  nationality, sex,
  COUNT (*) AS athlete_count
  FROM dim_athlete_clean
  GROUP BY nationality, sex
  ORDER BY nationality, sex
  LIMIT 20

""",conn)

,nationality,sex,athlete_count
0,None,None,1
1,None,Female,11079
2,None,Male,54047
3,Algeria,Male,10
4,Argentina,Male,2
5,Armenia,Female,1
6,Armenia,Male,33
7,Aruba,Male,4
8,Australia,Female,3
9,Australia,Male,56


Which (country, sex) combinations have more than 100 athletes?

In [59]:
pd.read_sql(
    """
    SELECT
      nationality,
      COUNT(*) AS athlete_count
    FROM dim_athlete_clean
    GROUP BY nationality
    HAVING COUNT(*) > 100;
    """,conn
)

,nationality,athlete_count
0,None,65127
1,Belarus,135
2,Croatia,328
3,Czechia,1115
4,East Germany,272
5,Russian Federation,1275
6,Serbia,307
7,Slovakia,224
8,Slovenia,132
9,Ukraine,327


Among MALE athletes only, which countries have more than 50 athletes?

In [60]:
pd.read_sql(
    """
    SELECT
      nationality,
      COUNT(*) AS athlete_count
    FROM dim_athlete_clean
    WHERE sex = 'Male'
    GROUP BY nationality
    HAVING COUNT(*) > 100;
    """,conn
)

,nationality,athlete_count
0,None,54047
1,Croatia,303
2,Czechia,929
3,East Germany,214
4,Russian Federation,1000
5,Serbia,250
6,Slovakia,163
7,Slovenia,116
8,Ukraine,220
9,West Germany,482


In [61]:
pd.read_sql(
    """
    SELECT
        name,
        type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY type, name;
    """,
    conn
)


,name,type
0,bios,table
1,bios_birth_final,table
2,bios_died_final,table
3,bios_measurements_final,table
4,dim_athlete,table
5,bios_birth_clean_v4,view
6,bios_cleaner,view
7,bios_died_clean,view
8,bios_measurements_clean,view
9,dim_athlete_clean,view


“Does any athlete appear more than once in dim_athlete?”
First uniqueness check (simplest)

In [62]:
pd.read_sql(
    """
    SELECT
      athlete_id,
      COUNT(*) AS athlete_count
    FROM dim_athlete
    GROUP BY athlete_id
    HAVING COUNT(*) > 1;
    """,conn
)

,athlete_id,athlete_count


Repeat uniqueness check for other tables
Birth table

In [63]:
pd.read_sql(
    """
    SELECT
      athlete_id,
      COUNT(*) AS athlete_count
    FROM bios_birth_final
    GROUP BY athlete_id
    HAVING COUNT(*) > 1;
    """,conn
)

,athlete_id,athlete_count


Repeat uniqueness check for other tables
bios measurements table

In [64]:
pd.read_sql(
    """
    SELECT
      athlete_id,
      COUNT(*) AS athlete_count
    FROM bios_measurements_final
    GROUP BY athlete_id
    HAVING COUNT(*) > 1;
    """,conn
)

,athlete_id,athlete_count


Repeat uniqueness check for other tables
Died table

In [65]:
pd.read_sql(
    """
    SELECT
      athlete_id,
      COUNT(*) AS athlete_count
    FROM bios_died_final
    GROUP BY athlete_id
    HAVING COUNT(*) > 1;
    """,conn
)

,athlete_id,athlete_count


Basic Null Count (foundation)

In [66]:
pd.read_sql("""SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN full_name IS NULL THEN 1 ELSE 0 END) AS null_full_name
FROM dim_athlete;
""",conn
)

,total_rows,null_full_name
0,70664,1


Basic Null Count (foundation)

In [67]:
pd.read_sql("""
SELECT *
FROM bios_birth_final;
""",conn
)

,athlete_id,born_raw,birth_date_raw,birth_date,birth_city,birth_region,birth_country_code
0,1.0,"12 December 1886 in Bordeaux, Gironde (FRA)",12 December 1886,1886-12-12,Bordeaux,Gironde,FRA
1,2.0,"1 April 1969 in Meulan, Yvelines (FRA)",1 April 1969,1969-04-01,Meulan,Yvelines,FRA
2,3.0,"13 August 1898 in Biarritz, Pyrénées-Atlantiqu...",13 August 1898,1898-08-13,Biarritz,Pyrénées-Atlantiques,FRA
3,4.0,"11 May 1895 in Paris VIIIe, Paris (FRA)",11 May 1895,1895-05-11,Paris VIIIe,Paris,FRA
4,5.0,"17 April 1878 in Wandsworth, England (GBR)",17 April 1878,1878-04-17,Wandsworth,England,GBR
...,...,...,...,...,...,...,...
56270,71187.0,"22 March 1878 in Budapest, Budapest (HUN)",22 March 1878,1878-03-22,Budapest,Budapest,HUN
56271,71188.0,"5 December 1947 in Tura, Pest (HUN)",5 December 1947,1947-12-05,Tura,Pest,HUN
56272,71189.0,"11 February 1911 in Budapest, Budapest (HUN)",11 February 1911,1911-02-11,Budapest,Budapest,HUN
56273,71190.0,"11 April 1871 in Budapest, Budapest (HUN)",11 April 1871,1871-04-11,Budapest,Budapest,HUN


In [68]:
pd.read_sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN birth_date IS NULL THEN 1 ELSE 0 END) AS null_count,
  ROUND(
    100.0 * SUM(CASE WHEN birth_date IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2)AS null_percent
FROM bios_birth_final;
""",conn
)

,total_rows,null_count,null_percent
0,56275,680,1.21


Enforcing a Threshold (PASS / FAIL logic)
Example rule:

“birth_country_code must be NULL in less than 1% of rows”

In [69]:
pd.read_sql("""
SELECT
    CASE
        WHEN
            100.0 * SUM(CASE WHEN birth_country_code IS NULL THEN 1 ELSE 0 END)
            / COUNT(*) < 1
        THEN 'PASS'
        ELSE 'FAIL'
    END AS quality_status
FROM bios_birth_final;

""",conn
    )

,quality_status
0,PASS


Multi-column Null Validation (real-world)

Example:

“Either height OR weight should exist”

In [70]:
pd.read_sql(
    """
    SELECT
      athlete_id
    FROM bios_measurements_final
    WHERE height_cm IS NULL
    AND weight_kg IS NULL;

    """,conn
)

,athlete_id
0,1.0
1,5.0
2,8.0
3,9.0
4,10.0
...,...
25275,71184.0
25276,71185.0
25277,71187.0
25278,71190.0


Why GROUP BY sometimes enters Null checks

Example:

“Which countries have unusually missing birth dates?”

In [71]:
pd.read_sql(
    """
    SELECT birth_country_code,
      COUNT(*) AS Total,
      SUM(CASE WHEN birth_date IS NULL THEN 1 ELSE 0 END) AS NULLS
    FROM bios_birth_final
    GROUP BY birth_country_code
    HAVING 100.0 * SUM(CASE WHEN birth_date IS NULL THEN 1 ELSE 0 END)/COUNT(*) > 10;
    """,conn
)

,birth_country_code,Total,NULLS
0,Alexandria,98,17
1,Athens,145,32
2,BRN,2,2
3,Bangalore,10,3
4,Basra,5,1
5,CGO,9,1
6,CHA,3,1
7,CYP,6,1
8,Cairo,159,24
9,Calcutta,25,4


Why GROUP BY sometimes enters Null checks

Example:

“Which countries have unusually missing birth dates?”

In [72]:
pd.read_sql(
    """
    SELECT sex, COUNT(*) AS sex
    FROM dim_athlete
    GROUP BY sex

    """,conn
)

,sex,sex
0,None,1
1,Female,12084
2,Male,58579


In [73]:
pd.read_sql(
    """
    SELECT * FROM dim_athlete
    LIMIT 5
    """,conn
)

,athlete_id,full_name,used_name,used_name_clean,sex,noc
0,1.0,"François Joseph Marie Antoine ""Jean-François""•...",Jean-François•Blanchy,Jean-François•Blanchy,Male,France
1,2.0,Arnaud Benjamin•Boetsch,Arnaud•Boetsch,Arnaud•Boetsch,Male,France
2,3.0,Jean Laurent Robert•Borotra,Jean•Borotra,Jean•Borotra,Male,France
3,4.0,Jacques Marie Stanislas Jean•Brugnon,Jacques•Brugnon,Jacques•Brugnon,Male,France
4,5.0,Henry Albert•Canet,Albert•Canet,Albert•Canet,Male,France


Which nationality values have more than 50 athletes?

In [74]:
pd.read_sql(
    """
    SELECT noc, COUNT(*) AS total_athletes
    FROM dim_athlete
    GROUP BY noc
    HAVING COUNT(*) > 50

    """,conn
)

,noc,total_athletes
0,Afghanistan,89
1,Algeria,178
2,Angola,74
3,Antigua and Barbuda,52
4,Argentina,1216
...,...,...
108,Venezuela,250
109,West Germany,1172
110,Yugoslavia,808
111,Zambia,84


Find all birth_country_code values where more than 10% of athletes have birth_date as NULL.

In [75]:
pd.read_sql(
    """
    SELECT birth_country_code,
      COUNT(*) AS Total,
      SUM(CASE WHEN birth_date IS NULL THEN 1 ELSE 0 END) AS NULLS
    FROM bios_birth_final
    GROUP BY birth_country_code
    HAVING 100.0 * SUM(CASE WHEN birth_date IS NULL THEN 1 ELSE 0 END)/COUNT(*) > 10;
    """,conn
)

,birth_country_code,Total,NULLS
0,Alexandria,98,17
1,Athens,145,32
2,BRN,2,2
3,Bangalore,10,3
4,Basra,5,1
5,CGO,9,1
6,CHA,3,1
7,CYP,6,1
8,Cairo,159,24
9,Calcutta,25,4


In [76]:
pd.read_sql(
    """
    SELECT * FROM dim_athlete
    LIMIT 5;
    """,conn
)

,athlete_id,full_name,used_name,used_name_clean,sex,noc
0,1.0,"François Joseph Marie Antoine ""Jean-François""•...",Jean-François•Blanchy,Jean-François•Blanchy,Male,France
1,2.0,Arnaud Benjamin•Boetsch,Arnaud•Boetsch,Arnaud•Boetsch,Male,France
2,3.0,Jean Laurent Robert•Borotra,Jean•Borotra,Jean•Borotra,Male,France
3,4.0,Jacques Marie Stanislas Jean•Brugnon,Jacques•Brugnon,Jacques•Brugnon,Male,France
4,5.0,Henry Albert•Canet,Albert•Canet,Albert•Canet,Male,France


INNER JOIN keeps only rows that exist in BOTH tables

If an athlete is missing from either side → row is dropped.

First **INNER JOIN** (no aggregates yet)

Question:
Show athlete name and birth date.

In [77]:
pd.read_sql(
    """
    SELECT
      a.athlete_id,
      a.full_name,
      b.birth_date
    FROM dim_athlete a
    INNER JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    LIMIT 10;
    """,conn
)

,athlete_id,full_name,birth_date
0,1.0,"François Joseph Marie Antoine ""Jean-François""•...",1886-12-12
1,2.0,Arnaud Benjamin•Boetsch,1969-04-01
2,3.0,Jean Laurent Robert•Borotra,1898-08-13
3,4.0,Jacques Marie Stanislas Jean•Brugnon,1895-05-11
4,5.0,Henry Albert•Canet,1878-04-17
5,6.0,Nicolas•Chatelain,1970-01-13
6,7.0,Patrick•Chila,1969-11-27
7,8.0,Henri Jean•Cochet,1901-12-14
8,9.0,Marcel•Cousin,1896-08-04
9,10.0,Luc Henri Hervé Guy•Gardye de la Chapelle,1868-07-16


In [78]:
pd.read_sql(
    """
    SELECT
      a.athlete_id,
      a.full_name,
      b.birth_date
    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    LIMIT 10
    """,conn
)

,athlete_id,full_name,birth_date
0,1.0,"François Joseph Marie Antoine ""Jean-François""•...",1886-12-12
1,2.0,Arnaud Benjamin•Boetsch,1969-04-01
2,3.0,Jean Laurent Robert•Borotra,1898-08-13
3,4.0,Jacques Marie Stanislas Jean•Brugnon,1895-05-11
4,5.0,Henry Albert•Canet,1878-04-17
5,6.0,Nicolas•Chatelain,1970-01-13
6,7.0,Patrick•Chila,1969-11-27
7,8.0,Henri Jean•Cochet,1901-12-14
8,9.0,Marcel•Cousin,1896-08-04
9,10.0,Luc Henri Hervé Guy•Gardye de la Chapelle,1868-07-16


“Show athlete profile with birth date (if available)” LEFT JOIN

In [79]:
pd.read_sql(
    """
    SELECT
      a.athlete_id,
      a.full_name,
      a.sex,
      a.noc,
      b.birth_date
    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    LIMIT 10;
    """,conn
)

,athlete_id,full_name,sex,noc,birth_date
0,1.0,"François Joseph Marie Antoine ""Jean-François""•...",Male,France,1886-12-12
1,2.0,Arnaud Benjamin•Boetsch,Male,France,1969-04-01
2,3.0,Jean Laurent Robert•Borotra,Male,France,1898-08-13
3,4.0,Jacques Marie Stanislas Jean•Brugnon,Male,France,1895-05-11
4,5.0,Henry Albert•Canet,Male,France,1878-04-17
5,6.0,Nicolas•Chatelain,Male,France,1970-01-13
6,7.0,Patrick•Chila,Male,France,1969-11-27
7,8.0,Henri Jean•Cochet,Male,France,1901-12-14
8,9.0,Marcel•Cousin,Male,France,1896-08-04
9,10.0,Luc Henri Hervé Guy•Gardye de la Chapelle,Male,France,1868-07-16


“Give me full athlete demographic + physical profile” Multiple Joins

In [80]:
pd.read_sql(
    """
    SELECT
      a.athlete_id,
      a.full_name,
      a.sex,
      a.noc,
      a.noc,

      b.birth_date,
      b.birth_country_code,

      m.height_cm,
      m.weight_kg,

      d.death_country_code

    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    LEFT JOIN bios_measurements_final m
      ON a.athlete_id = m.athlete_id
    LEFT JOIN bios_died_final d
      ON a.athlete_id = d.athlete_id
    LIMIT 10;
    """,conn
)

,athlete_id,full_name,sex,noc,noc,birth_date,birth_country_code,height_cm,weight_kg,death_country_code
0,1.0,"François Joseph Marie Antoine ""Jean-François""•...",Male,France,France,1886-12-12,FRA,NaN,NaN,FRA
1,2.0,Arnaud Benjamin•Boetsch,Male,France,France,1969-04-01,FRA,183.0,76.0,None
2,3.0,Jean Laurent Robert•Borotra,Male,France,France,1898-08-13,FRA,183.0,76.0,FRA
3,4.0,Jacques Marie Stanislas Jean•Brugnon,Male,France,France,1895-05-11,FRA,168.0,64.0,MON
4,5.0,Henry Albert•Canet,Male,France,France,1878-04-17,GBR,NaN,NaN,FRA
5,6.0,Nicolas•Chatelain,Male,France,France,1970-01-13,FRA,181.0,70.0,None
6,7.0,Patrick•Chila,Male,France,France,1969-11-27,FRA,180.0,73.0,None
7,8.0,Henri Jean•Cochet,Male,France,France,1901-12-14,FRA,NaN,NaN,FRA
8,9.0,Marcel•Cousin,Male,France,France,1896-08-04,FRA,NaN,NaN,FRA
9,10.0,Luc Henri Hervé Guy•Gardye de la Chapelle,Male,France,France,1868-07-16,FRA,NaN,NaN,None


VALIDATION AFTER JOINS

In [81]:
pd.read_sql(
    """
    SELECT
      COUNT(*) AS total_rows,
      COUNT(DISTINCT athlete_id) AS distinct_athletes
    FROM (
      SELECT
        a.athlete_id
      FROM dim_athlete a
      LEFT JOIN bios_birth_final b
        ON a.athlete_id = b.athlete_id
      LEFT JOIN bios_measurements_final m
        ON a.athlete_id = m.athlete_id
);

    """,conn
)

,total_rows,distinct_athletes
0,70664,70663


Which athletes exist in dim_athlete_final but have NO birth record?

In [82]:
pd.read_sql(
    """
    SELECT
      a.athlete_id,
      a.full_name,
      a.sex,
      a.noc,
      b.birth_date
    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    WHERE b.athlete_id IS NULL
    LIMIT 10;
    """,conn
)

,athlete_id,full_name,sex,noc,birth_date
0,13.0,J.•Defert,Male,France,None
1,14.0,Étienne•Durand,Male,France,None
2,28.0,Guy•Lejeune,Male,France,None
3,29.0,Albert•Lippmann,Male,France,None
4,109.0,Arthur B. J.•Norris,Male,Great Britain,None
5,165.0,Helen•Amankwah,Female,Ghana,None
6,166.0,Patricia Akosua•Offel,Female,Ghana,None
7,167.0,Patience Abena•Opokua,Female,Ghana,None
8,168.0,Winifred•Addy,Male,Ghana,None
9,170.0,Olga•Tsarmpopoulou,Female,Greece,None


Which athletes have NEITHER birth data NOR measurements?

In [83]:
pd.read_sql(
    """
    SELECT
      a.athlete_id,
      a.full_name

    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    LEFT JOIN bios_measurements_final m
      ON a.athlete_id = m.athlete_id
    WHERE b.athlete_id IS NULL
      AND m.athlete_id IS NULL
    LIMIT 10;
    """,conn
)

,athlete_id,full_name
0,None,None


In [84]:
pd.read_sql("""
SELECT
  COUNT(*) AS total_athletes,
    SUM(
        CASE
            WHEN b.athlete_id IS NULL
             AND m.athlete_id IS NULL
            THEN 1 ELSE 0
        END
    ) AS athletes_missing_both
FROM dim_athlete a
LEFT JOIN bios_birth_final b
    ON a.athlete_id = b.athlete_id
LEFT JOIN bios_measurements_final m
    ON a.athlete_id = m.athlete_id;
    """,conn)


,total_athletes,athletes_missing_both
0,70664,1


Count-based aggregations (foundation, but DE-style)
1.1 Athlete count by birth country

In [85]:
pd.read_sql(
    """
    SELECT
      b.birth_country_code,
      COUNT(*) AS athlete_count
    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    GROUP BY b.birth_country_code
    ORDER BY athlete_count DESC
    LIMIT 10;
    """,conn
)

,birth_country_code,athlete_count
0,None,14389
1,GER,4180
2,USA,4081
3,GBR,3947
4,FRA,3093
5,ITA,2226
6,CAN,2069
7,HUN,1793
8,SWE,1701
9,ESP,1522


1.2 Multi-dimensional count (country × sex)

In [86]:
pd.read_sql(
    """
    SELECT
      b.birth_country_code,
      a.sex,
      COUNT(*) AS athlete_count
    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    GROUP BY b.birth_country_code, a.sex
    ORDER BY b.birth_country_code, a.sex
    LIMIT 10;

    """,conn
)

,birth_country_code,sex,athlete_count
0,None,None,1
1,None,Female,2325
2,None,Male,12063
3,AFG,Male,17
4,AGU,Male,2
5,ALB,Female,5
6,ALB,Male,11
7,ALG,Female,7
8,ALG,Male,102
9,AND,Female,2


Metric aggregations (where mistakes usually happen)
2.1 Average height & weight by country

In [87]:
pd.read_sql(
    """
    SELECT
      b.birth_country_code,
      COUNT(*) AS athlete_count,
      ROUND(AVG(m.height_cm), 1) AS avg_height,
      ROUND(AVG(m.weight_kg), 1) AS avg_weight
    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    LEFT JOIN bios_measurements_final m
      ON a.athlete_id = m.athlete_id
    GROUP BY b.birth_country_code, a.sex
    ORDER BY avg_height DESC
    LIMIT 10;

    """,conn
)

,birth_country_code,athlete_count,avg_height,avg_weight
0,Milde,2,196.5,109.0
1,Sieg,1,196.0,101.0
2,Prignitz,1,194.0,90.0
3,Port Said,10,193.0,87.0
4,TOG,1,192.0,95.0
5,GIB,1,191.0,89.0
6,CHA,2,190.0,75.0
7,Dunkirk,1,189.0,95.0
8,MNE,35,188.3,86.6
9,Gonayiv,1,188.0,82.0


Threshold-based aggregation (REAL BUSINESS LOGIC)

Raw averages lie when sample size is small.

3.1 Apply minimum population rule

In [88]:
pd.read_sql(
    """
    SELECT
      b.birth_country_code,
      COUNT(*) AS ATHLETE_COUNT,
      ROUND(AVG(m.height_cm), 1) AS AVG_HEIGHT_CM,
      ROUND(AVG(m.weight_kg), 1) AS AVG_WEIGHT_KG
    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      on a.athlete_id = b.athlete_id
    LEFT JOIN bios_measurements_final m
      ON a.athlete_id = m.athlete_id
    GROUP BY b.birth_country_code
    HAVING COUNT(*) >= 50
    ORDER BY AVG_HEIGHT_CM, AVG_WEIGHT_KG
    LIMIT 10;
    """,conn
)

,birth_country_code,ATHLETE_COUNT,AVG_HEIGHT_CM,AVG_WEIGHT_KG
0,THA,50,168.5,60.5
1,DOM,64,169.8,65.5
2,ARM,62,170.0,69.0
3,ESA,70,170.1,64.2
4,COL,216,170.5,65.2
5,INA,233,170.6,65.8
6,MGL,89,170.7,70.1
7,PHI,155,171.1,65.1
8,TPE,60,171.1,67.6
9,JPN,1213,171.1,67.7


Conditional aggregation (VERY IMPORTANT)
4.1 Male vs Female counts in one query

In [89]:
pd.read_sql(
    """
    SELECT b.birth_country_code,
    SUM(CASE WHEN a.sex = 'Male' THEN 1 ELSE 0 END) AS MALE_COUNT,
    SUM(CASE WHEN a.sex = 'Female' THEN 1 ELSE 0 END) AS FEMALE_COUNT
    FROM dim_athlete a
    LEFT JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    GROUP BY b.birth_country_code
    ORDER BY b. birth_country_code
    LIMIT 10;
    """,conn
)

,birth_country_code,MALE_COUNT,FEMALE_COUNT
0,None,12063,2325
1,AFG,17,0
2,AGU,2,0
3,ALB,11,5
4,ALG,102,7
5,AND,8,2
6,ANG,28,2
7,ANT,5,1
8,ARG,704,47
9,ARM,58,4


Derived aggregations (age is NOT stored)
5.1 Average age by country

In [90]:
pd.read_sql(
    """
    SELECT
      b.birth_country_code,
      COUNT(*) AS athlete_count,
      ROUND(
          AVG(
              (julianday('now') - julianday(b.birth_date)) / 365.25
          ),
          1
      ) AS avg_age
  FROM dim_athlete a
  LEFT JOIN bios_birth_final b
    ON a.athlete_id = b.athlete_id
  WHERE b.birth_date IS NOT NULL
  GROUP BY b.birth_country_code
  HAVING COUNT(*) >= 50
  ORDER BY avg_age DESC
  LIMIT 10;

    """,conn
)

,birth_country_code,athlete_count,avg_age
0,Brussels,118,120.2
1,Antwerp,156,112.5
2,Ghent,96,111.4
3,Copenhagen,459,107.5
4,LUX,255,107.4
5,IRL,285,106.1
6,NOR,981,104.3
7,FRA,3087,104.1
8,Genoa,100,101.8
9,SWE,1701,101.4


SELECT
    b.birth_country_code,
    COUNT(*) AS athlete_count,
    ROUND(
        AVG(
            CASE
                WHEN d.death_date IS NOT NULL
                    THEN (julianday(d.death_date) - julianday(b.birth_date)) / 365.25
                ELSE
                    (julianday('now') - julianday(b.birth_date)) / 365.25
            END
        ),
        1
    ) AS avg_age
FROM dim_athlete_final a
LEFT JOIN bios_birth_final b
    ON a.athlete_id = b.athlete_id
LEFT JOIN bios_died_final d
    ON a.athlete_id = d.athlete_id
WHERE b.birth_date IS NOT NULL
GROUP BY b.birth_country_code
HAVING COUNT(*) >= 50
ORDER BY avg_age DESC;


Aggregation sanity validation (DO NOT SKIP)

Every aggregation must pass a reasonableness check.

In [91]:
pd.read_sql(
    """
SELECT
  SUM(athlete_count) AS summed_counts
FROM (
  SELECT
      b.birth_country_code,
      COUNT(*) AS athlete_count
  FROM dim_athlete a
  LEFT JOIN bios_birth_final b
    ON a.athlete_id = b.athlete_id
  GROUP BY b.birth_country_code
);

    """,conn
)

,summed_counts
0,70664


In [92]:
pd.read_sql(
    """
    SELECT * FROM bios_died_final
    LIMIT 5;
    """,conn
)

,athlete_id,died_raw,death_city,death_region,death_country_code
0,1.0,"2 October 1960 in Saint-Jean-de-Luz, Pyrénées-...",Saint-Jean-de-Luz,Pyrénées-Atlantiques,FRA
1,3.0,"17 July 1994 in Arbonne, Pyrénées-Atlantiques ...",Arbonne,Pyrénées-Atlantiques,FRA
2,4.0,"20 March 1978 in Monaco, Monaco (MON)",Monaco,Monaco,MON
3,5.0,"25 July 1930 in Paris VIIe, Paris (FRA)",Paris VIIe,Paris,FRA
4,8.0,"2 April 1987 in Saint-Germain-en-Laye, Yveline...",Saint-Germain-en-Laye,Yvelines,FRA


3.3 UNIQUENESS CHECKS (PROPERLY CLOSED)

We validate three kinds of uniqueness, not just athlete_id.

**3.3.1 Primary Key Uniqueness ✅ (Already DONE, but formalized)**

In [93]:
pd.read_sql(
"""
 SELECT
    athlete_id,
    COUNT(*) AS cnt
FROM dim_athlete
GROUP BY athlete_id
HAVING COUNT(*) > 1;
""",conn
)

,athlete_id,cnt


3.3.2 Natural Key Duplication ❌ → NOW DONE

**Natural keys represent real-world identity, not system IDs.

Example rule

No two athletes should share the same
(used_name_clean, birth_date, birth_country_code)**

In [94]:
pd.read_sql(
    """
    SELECT
        a.used_name_clean,
        b.birth_date,
        b.birth_country_code,
        COUNT(*) AS cnt
    FROM dim_athlete a
    JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    GROUP BY
      a.used_name_clean,
      b.birth_date,
      b.birth_country_code
    HAVING COUNT(*) > 1
    """,conn
)

,used_name_clean,birth_date,birth_country_code,cnt


3.3.3 Composite Uniqueness ❌ → NOW DONE

Composite keys assert row-level integrity.

Rule

(athlete_id, birth_country_code) must be unique in birth table

In [95]:
pd.read_sql(
    """
    SELECT
      athlete_id,
      birth_country_code,
      COUNT(*) AS cnt
    FROM bios_birth_final
    GROUP BY athlete_id, birth_country_code
    HAVING COUNT(*) > 1;
    """,conn
)


,athlete_id,birth_country_code,cnt


3.4 REFERENTIAL INTEGRITY CHECKS (NOW DONE)

This ensures no orphaned records and no broken joins.

3.4.1 Orphan Records ❌ → NOW DONE
Rule

**Every record in child tables must exist in dim_athlete_final**

**Birth table orphan check**

In [96]:
pd.read_sql(
    """
    SELECT b.athlete_id
    FROM bios_birth_final b
    LEFT JOIN dim_athlete a
    ON a.athlete_id = b.athlete_id
WHERE a.athlete_id IS NULL;

    """,conn
)


,athlete_id


Measurements table orphan check

In [97]:
pd.read_sql(
    """
    SELECT m.athlete_id
    FROM bios_measurements_final m
    LEFT JOIN dim_athlete a
      ON a.athlete_id = m.athlete_id
    WHERE a.athlete_id IS NULL;

    """,conn
)


,athlete_id
0,None


Died table orphan check

In [98]:
pd.read_sql(
    """
    SELECT d.athlete_id
FROM bios_died_final d
LEFT JOIN dim_athlete a
    ON a.athlete_id = d.athlete_id
WHERE a.athlete_id IS NULL;

    """,conn
)

,athlete_id


3.4.2 Broken Join Detection ❌ → NOW DONE
Rule

Joining dimensions must not create or lose records unintentionally

In [99]:
pd.read_sql(
    """
    SELECT d.athlete_id
FROM bios_died_final d
LEFT JOIN dim_athlete a
    ON a.athlete_id = d.athlete_id
WHERE a.athlete_id IS NULL;

    """,conn
)

,athlete_id


3.5.1 Temporal Logic ❌ → NOW DONE
Rule

birth_date < death_date (when death_date exists)
✔ Prevents impossible timelines.

In [100]:
pd.read_sql(
    """
    SELECT
    b.athlete_id,
    b.birth_date,
    d.death_date
FROM bios_birth_final b
JOIN bios_died_final d
    ON b.athlete_id = d.athlete_id
WHERE d.death_date <= b.birth_date;

    """,conn
)


DatabaseError: Execution failed on sql '
    SELECT
    b.athlete_id,
    b.birth_date,
    d.death_date
FROM bios_birth_final b
JOIN bios_died_final d
    ON b.athlete_id = d.athlete_id
WHERE d.death_date <= b.birth_date;

    ': no such column: d.death_date

3.5.2 Domain Constraints ❌ → NOW DONE
Height must be realistic (example: 120–250 cm)

In [101]:
pd.read_sql(
    """SELECT athlete_id, height_cm
FROM bios_measurements_final
WHERE height_cm IS NOT NULL
  AND (height_cm < 120 OR height_cm > 250);
""",conn)

,athlete_id,height_cm


3.5.3 Weight must be realistic (example: 30–300 kg)

In [102]:
pd.read_sql(
"""
SELECT athlete_id, weight_kg
FROM bios_measurements_final
WHERE weight_kg IS NOT NULL
  AND (weight_kg < 30 OR weight_kg > 300);

""",conn)

,athlete_id,weight_kg
0,28926.0,28
1,28976.0,25


MODULE — WINDOW FUNCTIONS (DATA ENGINEERING LEVEL)
What problem do Window Functions solve?
GROUP BY limitation

When you use GROUP BY:

Rows are collapsed

You lose row-level detail

But many business questions need:

Row-level data AND

Group-level context

That’s exactly what window functions provide.

Core Definition (memorize this)

A window function performs a calculation across a set of related rows
without collapsing them into a single row.

In [103]:
pd.read_sql(
    """
    SELECT
      a.athlete_id,
      a.full_name,
      b.birth_country_code,
      ROW_NUMBER() OVER(
        PARTITION BY b.birth_country_code
        ORDER BY a.athlete_id
      )AS row_number
    FROM dim_athlete a
    JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    LIMIT 50;
    """,conn

  )

,athlete_id,full_name,birth_country_code,row_number
0,57056.0,Mohammad•Ebrahimi,AFG,1
1,57062.0,Kayum•Ayub,AFG,2
2,57064.0,Aka-Jahan•Dastagir,AFG,3
3,57065.0,Ghulam•Dastagir,AFG,4
4,57066.0,Ahmad•Djan,AFG,5
5,57068.0,Sultan Mohammad•Dost,AFG,6
6,57069.0,Ghulam Mohiddin•Gunga,AFG,7
7,57072.0,Mohammad Ibrahim•Kederi,AFG,8
8,57073.0,Faiz Mohammad•Khakshar,AFG,9
9,57074.0,Amir Jan•Khalunder,AFG,10


Business question

“Give me the Top 3 tallest athletes per country.”

In [104]:
pd.read_sql(
    """
    SELECT *
FROM (
    SELECT
        a.full_name,
        b.birth_country_code,
        m.height_cm,
        DENSE_RANK() OVER (
            PARTITION BY b.birth_country_code
            ORDER BY m.height_cm DESC
        ) AS height_rank
    FROM dim_athlete a
    JOIN bios_birth_final b
        ON a.athlete_id = b.athlete_id
    JOIN bios_measurements_final m
        ON a.athlete_id = m.athlete_id
    WHERE m.height_cm IS NOT NULL
)
WHERE height_rank <= 3
ORDER BY birth_country_code, height_rank
LIMIT 50;
    """,conn
)

,full_name,birth_country_code,height_cm,height_rank
0,Ghulam•Dastagir,AFG,180,1
1,Abdul Hakim•Wardak,AFG,179,2
2,Abdul Hadi•Shekaib,AFG,178,3
3,Keith Leroy•Connor,AGU,186,1
4,Earl Anthony•Richardson,AGU,179,2
5,Besnik•Musaj,ALB,183,1
6,Shkëlqim•Troplini,ALB,180,2
7,Vera•Bitanji (Bregu-),ALB,178,3
8,Abdel Krim•Ben Djemil,ALG,190,1
9,Frédéric•Perez,ALG,190,1


Business Question (REAL, from your data)

For each athlete, show:

their height

the average height of athletes from their birth country

and how much they differ from that average

In [105]:
pd.read_sql(
    """
    SELECT
    a.athlete_id,
    a.full_name,
    b.birth_country_code,
    m.height_cm,

    -- country-level average height
    ROUND(
        AVG(m.height_cm) OVER (
            PARTITION BY b.birth_country_code
        ),
        1
    ) AS country_avg_height

FROM dim_athlete a
JOIN bios_birth_final b
    ON a.athlete_id = b.athlete_id
JOIN bios_measurements_final m
    ON a.athlete_id = m.athlete_id

WHERE m.height_cm IS NOT NULL
LIMIT 50;

    """,conn
)

,athlete_id,full_name,birth_country_code,height_cm,country_avg_height
0,57056.0,Mohammad•Ebrahimi,AFG,160,169.8
1,57062.0,Kayum•Ayub,AFG,176,169.8
2,57064.0,Aka-Jahan•Dastagir,AFG,170,169.8
3,57065.0,Ghulam•Dastagir,AFG,180,169.8
4,57066.0,Ahmad•Djan,AFG,165,169.8
5,57068.0,Sultan Mohammad•Dost,AFG,168,169.8
6,57069.0,Ghulam Mohiddin•Gunga,AFG,168,169.8
7,57072.0,Mohammad Ibrahim•Kederi,AFG,166,169.8
8,57073.0,Faiz Mohammad•Khakshar,AFG,162,169.8
9,57074.0,Amir Jan•Khalunder,AFG,166,169.8


Compare athlete vs country average (REAL VALUE)

Now let’s add difference from average.

In [106]:
pd.read_sql(
    """
    SELECT
      a.athlete_id,
      a.full_name,
      b.birth_country_code,
      m.height_cm,
      ROUND(
          AVG(m.height_cm) OVER(
            PARTITION BY b.birth_country_code
          ), 1
      )AS AVG_COUNTRY_HEIGHT,
      ROUND(
        m.height_cm
        - AVG(m.height_cm) OVER(
          PARTITION BY b.birth_country_code
        ), 1
      )AS DIFFERECE_FROM_COUNTRY_AVG
    FROM dim_athlete a
    JOIN bios_birth_final b
      ON a.athlete_id = b.athlete_id
    JOIN bios_measurements_final m
      ON a.athlete_id = m.athlete_id
    WHERE m.height_cm IS NOT NULL
    LIMIT 50;
    """,conn
)

,athlete_id,full_name,birth_country_code,height_cm,AVG_COUNTRY_HEIGHT,DIFFERECE_FROM_COUNTRY_AVG
0,57056.0,Mohammad•Ebrahimi,AFG,160,169.8,-9.8
1,57062.0,Kayum•Ayub,AFG,176,169.8,6.2
2,57064.0,Aka-Jahan•Dastagir,AFG,170,169.8,0.2
3,57065.0,Ghulam•Dastagir,AFG,180,169.8,10.2
4,57066.0,Ahmad•Djan,AFG,165,169.8,-4.8
5,57068.0,Sultan Mohammad•Dost,AFG,168,169.8,-1.8
6,57069.0,Ghulam Mohiddin•Gunga,AFG,168,169.8,-1.8
7,57072.0,Mohammad Ibrahim•Kederi,AFG,166,169.8,-3.8
8,57073.0,Faiz Mohammad•Khakshar,AFG,162,169.8,-7.8
9,57074.0,Amir Jan•Khalunder,AFG,166,169.8,-3.8


For each athlete, show how many athletes are there from their birth country.

In [107]:
pd.read_sql(
    """
    SELECT
    a.athlete_id,
    a.full_name,
    b.birth_country_code,

    COUNT(*) OVER (
        PARTITION BY b.birth_country_code
    ) AS athletes_in_country

FROM dim_athlete a
JOIN bios_birth_final b
    ON a.athlete_id = b.athlete_id;

    """,conn
)

,athlete_id,full_name,birth_country_code,athletes_in_country
0,57056.0,Mohammad•Ebrahimi,AFG,17
1,57062.0,Kayum•Ayub,AFG,17
2,57064.0,Aka-Jahan•Dastagir,AFG,17
3,57065.0,Ghulam•Dastagir,AFG,17
4,57066.0,Ahmad•Djan,AFG,17
...,...,...,...,...
56270,51767.0,Evan•Stewart,ZIM,38
56271,61703.0,Anthony James•Crossley,ZIM,38
56272,62956.0,Michael•McFadden,ZIM,38
56273,24901.0,"Luis ""Kiriki""•Iruretagoyena Ayestarán",Zarautz,2


For each athlete, show the total combined height of athletes from their country.

In [108]:
pd.read_sql(
    """
    SELECT
    a.athlete_id,
    a.full_name,
    b.birth_country_code,
    m.height_cm,

    SUM(m.height_cm) OVER (
        PARTITION BY b.birth_country_code
    ) AS total_country_height

FROM dim_athlete a
JOIN bios_birth_final b
    ON a.athlete_id = b.athlete_id
JOIN bios_measurements_final m
    ON a.athlete_id = m.athlete_id

WHERE m.height_cm IS NOT NULL;

    """,conn
)

,athlete_id,full_name,birth_country_code,height_cm,total_country_height
0,57056.0,Mohammad•Ebrahimi,AFG,160,2887
1,57062.0,Kayum•Ayub,AFG,176,2887
2,57064.0,Aka-Jahan•Dastagir,AFG,170,2887
3,57065.0,Ghulam•Dastagir,AFG,180,2887
4,57066.0,Ahmad•Djan,AFG,165,2887
...,...,...,...,...,...
36842,49979.0,"Amanda Toni ""Mandy""•Loots",ZIM,168,5735
36843,51760.0,Antonette•Wilken (-Batchelor),ZIM,162,5735
36844,51767.0,Evan•Stewart,ZIM,177,5735
36845,61703.0,Anthony James•Crossley,ZIM,170,5735


“One athlete per country (using ROW_NUMBER)” **USING CTEs**

In [109]:
pd.read_sql(
    """
    WITH numbered_athletes AS(
      SELECT
        a.full_name,
        b.birth_country_code,
        ROW_NUMBER() OVER (
            PARTITION BY b.birth_country_code
            ORDER BY a.athlete_id
        ) AS rn
    FROM dim_athlete a
    JOIN bios_birth_final b
        ON a.athlete_id = b.athlete_id
    )
    SELECT *
    FROM numbered_athletes
    WHERE rn=1;

    """,conn
)

,full_name,birth_country_code,rn
0,Mohammad•Ebrahimi,AFG,1
1,Earl Anthony•Richardson,AGU,1
2,Besnik•Musaj,ALB,1
3,Patrick•Birocheau,ALG,1
4,Emili•Pérez Font,AND,1
...,...,...,...
329,Slobodan•Mišković,YUG,1
330,Albert Robert Corneille•Muylle,Ypres,1
331,Ian William•McLoughlin,ZAM,1
332,Robin David•Sampson,ZIM,1


Focus Cleaning CRM

Client’s Response to the Requirement Queries **USING CTEs**

In [110]:
pd.read_sql(
    """
    WITH base_athletes AS (
    SELECT
        a.athlete_id,
        a.full_name,
        b.birth_country_code,
        m.height_cm
    FROM dim_athlete a
    JOIN bios_birth_final b
        ON a.athlete_id = b.athlete_id
    JOIN bios_measurements_final m
        ON a.athlete_id = m.athlete_id
    WHERE m.height_cm IS NOT NULL
)
, ranked_athletes AS (
    SELECT
        *,
        DENSE_RANK() OVER (
            PARTITION BY birth_country_code
            ORDER BY height_cm DESC
        ) AS height_rank
    FROM base_athletes
)
SELECT
    full_name,
    birth_country_code,
    height_cm
FROM ranked_athletes
WHERE height_rank = 1;


    """,conn
)

,full_name,birth_country_code,height_cm
0,Ghulam•Dastagir,AFG,180
1,Keith Leroy•Connor,AGU,186
2,Besnik•Musaj,ALB,183
3,Abdel Krim•Ben Djemil,ALG,190
4,Frédéric•Perez,ALG,190
...,...,...,...
344,Mohamed Mahfood•Sayed,YMD,175
345,Slobodan•Mišković,YUG,185
346,Edsard Frederik•Schlingemann,ZAM,187
347,"Johannes ""Hans""•Lamprecht",ZIM,187


Business Question (realistic & meaningful)

For each athlete, show:

their height

their country

the average height of their country

and how they rank within their country by height

**USING CTEs**

In [111]:
pd.read_sql(
    """
    WITH base_athletes AS (
    SELECT
        a.athlete_id,
        a.full_name,
        b.birth_country_code,
        m.height_cm
    FROM dim_athlete a
    JOIN bios_birth_final b
        ON a.athlete_id = b.athlete_id
    JOIN bios_measurements_final m
        ON a.athlete_id = m.athlete_id
    WHERE m.height_cm IS NOT NULL
)
, country_aggregates AS (
    SELECT
        birth_country_code,
        AVG(height_cm) AS avg_country_height,
        COUNT(*) AS athlete_count
    FROM base_athletes
    GROUP BY birth_country_code
)
, athlete_enriched AS (
    SELECT
        b.athlete_id,
        b.full_name,
        b.birth_country_code,
        b.height_cm,

        c.avg_country_height,
        c.athlete_count,

        DENSE_RANK() OVER (
            PARTITION BY b.birth_country_code
            ORDER BY b.height_cm DESC
        ) AS height_rank

    FROM base_athletes b
    JOIN country_aggregates c
        ON b.birth_country_code = c.birth_country_code
)
SELECT *
FROM athlete_enriched;

    """,conn
)

,athlete_id,full_name,birth_country_code,height_cm,avg_country_height,athlete_count,height_rank
0,57065.0,Ghulam•Dastagir,AFG,180,169.823529,17,1
1,64263.0,Abdul Hakim•Wardak,AFG,179,169.823529,17,2
2,64261.0,Abdul Hadi•Shekaib,AFG,178,169.823529,17,3
3,57062.0,Kayum•Ayub,AFG,176,169.823529,17,4
4,64257.0,Abdul Ghafar•Ghafoori,AFG,172,169.823529,17,5
...,...,...,...,...,...,...,...
36842,51760.0,Antonette•Wilken (-Batchelor),ZIM,162,173.787879,33,14
36843,49918.0,"Alexandra ""Sandra""•Morgenrood",ZIM,161,173.787879,33,15
36844,20454.0,Maureen Jean•George,ZIM,158,173.787879,33,16
36845,49914.0,"Susarah Jacoba Elizabeth ""Sarie""•Bezuidenhout ...",ZIM,158,173.787879,33,16


Business Question (realistic & meaningful)

For each athlete, show:

their height

their country

the average height of their country

and how they rank within their country by height

**Combining 2 CTEs**

In [112]:
pd.read_sql(
    """
    WITH base_athletes AS (
    SELECT
        a.athlete_id,
        a.full_name,
        b.birth_country_code,
        m.height_cm
    FROM dim_athlete a
    JOIN bios_birth_final b
        ON a.athlete_id = b.athlete_id
    JOIN bios_measurements_final m
        ON a.athlete_id = m.athlete_id
    WHERE m.height_cm IS NOT NULL
)
, country_aggregates AS (
    SELECT
        birth_country_code,
        AVG(height_cm) AS avg_country_height,
        COUNT(*) AS athlete_count
    FROM base_athletes
    GROUP BY birth_country_code
)
, athlete_enriched AS (
    SELECT
        b.athlete_id,
        b.full_name,
        b.birth_country_code,
        b.height_cm,

        c.avg_country_height,
        c.athlete_count,

        DENSE_RANK() OVER (
            PARTITION BY b.birth_country_code
            ORDER BY b.height_cm DESC
        ) AS height_rank

    FROM base_athletes b
    JOIN country_aggregates c
        ON b.birth_country_code = c.birth_country_code
)
SELECT
    full_name,
    birth_country_code,
    height_cm,
    avg_country_height
FROM athlete_enriched
WHERE height_rank = 1;

    """,conn
)

,full_name,birth_country_code,height_cm,avg_country_height
0,Ghulam•Dastagir,AFG,180,169.823529
1,Keith Leroy•Connor,AGU,186,182.500000
2,Besnik•Musaj,ALB,183,170.600000
3,Abdel Krim•Ben Djemil,ALG,190,174.539474
4,Frédéric•Perez,ALG,190,174.539474
...,...,...,...,...
344,Mohamed Mahfood•Sayed,YMD,175,175.000000
345,Slobodan•Mišković,YUG,185,184.000000
346,Edsard Frederik•Schlingemann,ZAM,187,173.000000
347,"Johannes ""Hans""•Lamprecht",ZIM,187,173.787879
